# Dunnhumby K=1 CLV 가치기저 M5 네 모형 screen
원 논문 BPR(양성당 음성 1개)로 M1·M2·M4·M5를 같은 실행에서 학습합니다. M2는 q_C로 조절한 q_V 가격위치 기저(ρ=0.25)를 ID 전파 뒤에 붙입니다. seed 42, DAY 1~683 학습 → 684~690 개발평가, 100 epoch. test·holdout은 만들지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = 'd0c9aec0f0f1f903a01ccdf207b41088d4ac75fc'
%cd /content
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA
import subprocess
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() == REVIEWED_SHA


In [ ]:
import json
import torch
import lightgcn_clv_m5_clv_scaled_value_basis_k1_screen as screen

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
assert screen.CODE_VERSION == 'm5-clv-scaled-value-basis-k1-four-arm-screen-v1'
cfg = screen.configure_value_basis_k1_screen()
summary = screen.preflight_summary(cfg)
assert summary['loss']['negative_count'] == 1
assert summary['m2']['rho'] == 0.25
assert summary['m2']['q_c_used_in_m2'] is True
assert summary['m2']['economic_graph_propagation'] is False
assert summary['reused_models'] == []
print(json.dumps(summary, ensure_ascii=False, indent=2))


In [ ]:
result_df = screen.run_value_basis_k1_screen(cfg)


In [ ]:
import pandas as pd
from IPython.display import display

def show(frame):
    view = frame.copy()
    view.attrs = {}
    display(view)

core = ['model_id', 'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20', 'recall@50', 'ndcg@50',
        'price_purchase_amount_weighted_hit@10', 'vndcg@10', 'coverage@10']
print('1) 네 모형 핵심 절대지표')
show(result_df[core])
print('2) Top-10 목록 변경 비율 (지표 해석 전 먼저 확인)')
show(result_df.attrs['top10_overlap'])
print('3) 경제점수 영향력')
show(result_df.attrs['score_diagnostics'])
print('4) 판독')
print(json.dumps(result_df.attrs['decision'], ensure_ascii=False, indent=2))
print('5) 전체 비교표 파일:', result_df.attrs['result_paths']['comparison_csv'])
print('저장 파일:', json.dumps(result_df.attrs['result_paths'], ensure_ascii=False, indent=2))
